# learn-to-remember: M3 probe + baselines (Kaggle, GPU T4 x2)

Settings: **Accelerator = GPU T4 x2**, **Internet = on**. Run the cells top to bottom.

What this measures (nothing here is estimated):
1. `logprob_memory`: peak memory of one training-style log-prob pass at 2k/4k context (checks the M3 logit-chunking fix).
2. Model probe: Qwen2.5-1.5B-Instruct and Qwen3-1.7B (non-thinking) on the 50 probe questions, notes v3, greedy. Rule: valid tool calls >= 80% and accuracy in [20%, 60%].
3. Timing: seconds per question for the chosen model on 32 questions.
4. Baselines: chosen model, notes v3 and no notes, on company A (293 q) and company B (300 q), one company per GPU.

At the end, download `/kaggle/working/m3_results.zip` and bring it back.

In [ ]:
import os, subprocess, sys
REPO = "/kaggle/working/learn-to-remember"
# Option A (needs the M3 commit on GitHub): clone.
# Option B: upload the repo folder as a Kaggle dataset named "learn-to-remember" and it is copied instead.
if not os.path.exists(REPO):
    if os.path.exists("/kaggle/input/learn-to-remember"):
        subprocess.run(["cp", "-r", "/kaggle/input/learn-to-remember", REPO], check=True)
    else:
        subprocess.run(["git", "clone", "https://github.com/Vivek-Chaudhari30/learn-to-remember.git", REPO], check=True)
os.chdir(REPO)
print(subprocess.run(["git", "log", "--oneline", "-3"], capture_output=True, text=True).stdout)

In [ ]:
# Keep Kaggle's CUDA torch. Qwen3 needs transformers >= 4.51.
!pip install -q "transformers>=4.51,<6" "peft>=0.12" "accelerate>=0.34" pyyaml
import torch, transformers, peft
print(torch.__version__, transformers.__version__, peft.__version__, torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
# Fetch both tokenizers (small) so the tokenizer-dependent tests run instead of skipping, then run the suite.
from transformers import AutoTokenizer
for m in ["Qwen/Qwen2.5-0.5B-Instruct", "Qwen/Qwen3-1.7B"]:
    AutoTokenizer.from_pretrained(m)
!python -m pytest -q 2>&1 | tail -3

## 1. Log-prob memory (one 1.5B row, LoRA + gradient checkpointing, fp16)

In [ ]:
!python scripts/logprob_memory.py --model Qwen/Qwen2.5-1.5B-Instruct --dtype fp16 --lengths 2048,4096 --out results/memory

## 2. Model probe (50 questions, company A, notes v3, greedy)

In [ ]:
!python scripts/probe_models.py --dtype fp16 --models Qwen/Qwen2.5-1.5B-Instruct,Qwen/Qwen3-1.7B --out results/probe 2>&1 | grep -v Warning

## 3. Choose the model and time it

Set `CHOSEN` from the probe output (the §3.5 rule). If both pass, keep the spec default (Qwen2.5-1.5B-Instruct).
If neither passes, stop here and bring back the probe results: that decision (e.g. an SFT warm start) is made outside this notebook.

In [ ]:
CHOSEN = "Qwen/Qwen2.5-1.5B-Instruct"
SLUG = CHOSEN.split("/")[-1].lower()
!python scripts/run_eval.py --model {CHOSEN} --dtype fp16 --companies company_a --conditions v3 --limit 32 --n-boot 200 --out results/timing/{SLUG} 2>&1 | grep -v Warning | tail -8

## 4. Baselines: company A on GPU 0, company B on GPU 1, in parallel

Each process runs notes v3 then no notes. The seconds-per-question from step 3 tells you how long this takes
(about 2 x 300 x s/question per process). Progress goes to the two log files; the cell waits for both.

In [ ]:
import subprocess, time
procs = []
for gpu, company in [(0, "company_a"), (1, "company_b")]:
    log = open(f"/kaggle/working/baseline_{company}.log", "w")
    cmd = ["python", "-u", "scripts/run_eval.py", "--model", CHOSEN, "--dtype", "fp16",
           "--companies", company, "--conditions", "v3,no_notes",
           "--out", f"results/baselines/{SLUG}/{company}"]
    procs.append(subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT,
                                  env={**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}))
t0 = time.time()
while any(p.poll() is None for p in procs):
    time.sleep(120)
    for company in ["company_a", "company_b"]:
        lines = [l for l in open(f"/kaggle/working/baseline_{company}.log") if "s/question" in l]
        print(f"[{(time.time()-t0)/60:.0f} min] {company}: {lines[-1].strip() if lines else '...'}")
print("exit codes:", [p.returncode for p in procs])

In [ ]:
for company in ["company_a", "company_b"]:
    print(open(f"results/baselines/{SLUG}/{company}/tables.md").read())

## Optional: base model on notes v0-v2 (company A), the drift baseline for RQ3

Only if there is session time left; the timing above tells you how long it takes.

In [ ]:
# !CUDA_VISIBLE_DEVICES=0 python scripts/run_eval.py --model {CHOSEN} --dtype fp16 --companies company_a --conditions v0,v1,v2 --out results/baselines/{SLUG}/company_a_drift

## Package the results

In [ ]:
!cp /kaggle/working/baseline_*.log results/ 2>/dev/null; cd /kaggle/working/learn-to-remember && zip -qr /kaggle/working/m3_results.zip results/memory results/probe results/timing results/baselines results/baseline_*.log && ls -lh /kaggle/working/m3_results.zip